# RedlineIQ Vehicle Dimensions EDA + ETL

**Inputs used:**
- `vehicles.csv`
- `vehicle_dimensions.csv`
- `vehicle_chassis_specs.csv`
- `vehicle_powertrain_specs.csv`
- `vehicle_factory_performance.csv`

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re
import json

# Helper functions

In [23]:
# EDA AND ETL RAG/AI

def stable_id(*values, prefix: str = "") -> str:
    raw = "|".join("" if v is None or pd.isna(v) else str(v) for v in values)
    digest = hashlib.md5(raw.encode("utf-8")).hexdigest()[:14]
    return f"{prefix}{digest}"


def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    return None


def first_existing(input_dir: Path, candidates: Iterable[str]) -> Path | None:
    for name in candidates:
        path = input_dir / name
        if path.exists():
            return path
    return None


def clean_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip("|").strip()
    text = re.sub(r"\s*\|\s*", " | ", text)
    return text if text else np.nan


def clean_frame_text(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].apply(clean_text)
    return df


def numericize_non_id_columns(df: pd.DataFrame, skip_cols: set[str] | None = None) -> pd.DataFrame:
    df = df.copy()
    skip_cols = skip_cols or set()
    for col in df.columns:
        if col not in skip_cols and df[col].dtype == "object":
            converted = pd.to_numeric(df[col], errors="coerce")
            # Only replace when conversion has at least one value and does not erase mostly text.
            if converted.notna().sum() > 0 and converted.notna().sum() >= df[col].notna().sum() * 0.8:
                df[col] = converted
    return df


def prefix_non_key_columns(df: pd.DataFrame, prefix: str, key: str = "vehicle_id") -> pd.DataFrame:
    rename_map = {c: f"{prefix}_{c}" for c in df.columns if c != key}
    return df.rename(columns=rename_map)


def dedupe_by_vehicle_id(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or "vehicle_id" not in df.columns:
        return df
    df = df.copy()
    df["_non_null_count"] = df.notna().sum(axis=1)
    df = (
        df.sort_values(["vehicle_id", "_non_null_count"], ascending=[True, False])
        .drop_duplicates(subset=["vehicle_id"], keep="first")
        .drop(columns=["_non_null_count"])
        .reset_index(drop=True)
    )
    return df


def classify_fact_use_case(fact_key: str, fact_label: str, section_name: str) -> str:
    text = f"{fact_key} {fact_label} {section_name}".lower()

    if "suspension" in text:
        return "suspension_setup_context"
    if "tyre" in text or "tire" in text or "rim" in text:
        return "wheel_tire_fitment_context"
    if "brake" in text:
        return "brake_upgrade_context"
    if "fuel" in text or "tank" in text or "consumption" in text:
        return "fueling_range_context"
    if "oil" in text or "lubrication" in text:
        return "maintenance_oil_context"
    if "engine" in text or "horsepower" in text or "torque" in text or "compression" in text or "aspiration" in text:
        return "powertrain_context"
    if "weight" in text or "dimensions" in text or "aerodynamics" in text or "drag" in text:
        return "dimensions_aero_weight_context"
    if "emission" in text or "catalytic" in text or "co2" in text:
        return "emissions_context"
    if "performance" in text or "speed" in text or "acceleration" in text:
        return "factory_performance_context"

    return "general_vehicle_context"


def build_fact_statement(row: pd.Series) -> str:
    vehicle_id = row.get("vehicle_id", "")
    section = row.get("section_name", "")
    label = row.get("fact_label", "")
    value = row.get("fact_value", "")
    source = row.get("source_name", "")
    return f"Vehicle {vehicle_id} | Section: {section} | {label}: {value} | Source: {source}".strip()


def load_inputs(input_dir: Path) -> dict[str, pd.DataFrame | None]:
    chassis_path = first_existing(input_dir, ["vehicle_chassis_specs.csv", "vehicle_chassis_specs(1).csv"])

    paths = {
        "vehicles": input_dir / "vehicles.csv",
        "dimensions": input_dir / "vehicle_dimensions.csv",
        "chassis": chassis_path,
        "powertrain": input_dir / "vehicle_powertrain_specs.csv",
        "performance": input_dir / "vehicle_factory_performance.csv",
        "fuel_emissions": input_dir / "vehicle_fuel_emissions.csv",
        "source_facts": input_dir / "vehicle_source_facts.csv",
    }

    return {name: read_csv_if_exists(path) if path is not None else None for name, path in paths.items()}



In [24]:
def build_wide_vehicle_profile(inputs: dict[str, pd.DataFrame | None]) -> pd.DataFrame:
    """Combine all structured spec tables into one wide vehicle profile table."""
    table_order = [
        ("vehicles", "vehicle"),
        ("dimensions", "dim"),
        ("chassis", "chassis"),
        ("powertrain", "powertrain"),
        ("performance", "perf"),
        ("fuel_emissions", "fuel"),
    ]

    profile = None

    for table_name, prefix in table_order:
        df = inputs.get(table_name)
        if df is None:
            continue

        df = clean_frame_text(df)
        df = numericize_non_id_columns(df, skip_cols={"vehicle_id"})
        df = dedupe_by_vehicle_id(df)

        # Keep vehicles columns readable. Prefix all domain tables to avoid losing fields with same names.
        if table_name != "vehicles":
            df = prefix_non_key_columns(df, prefix=prefix, key="vehicle_id")

        if profile is None:
            profile = df
        else:
            profile = profile.merge(df, on="vehicle_id", how="outer")

    if profile is None:
        raise FileNotFoundError("No structured vehicle files were found.")

    # Useful cross-table derived features.
    if {"dim_curb_weight_lb", "powertrain_horsepower_hp"}.issubset(profile.columns):
        profile["derived_lb_per_hp"] = profile["dim_curb_weight_lb"] / profile["powertrain_horsepower_hp"].replace({0: np.nan})
        profile["derived_hp_per_1000_lb"] = (profile["powertrain_horsepower_hp"] * 1000) / profile["dim_curb_weight_lb"].replace({0: np.nan})

    if {"dim_length_in", "dim_width_in"}.issubset(profile.columns):
        profile["derived_footprint_sq_ft"] = (profile["dim_length_in"] * profile["dim_width_in"]) / 144

    if {"dim_width_in", "dim_height_in"}.issubset(profile.columns):
        profile["derived_frontal_area_proxy_sq_ft"] = (profile["dim_width_in"] * profile["dim_height_in"]) / 144

    if {"dim_wheelbase_in", "dim_length_in"}.issubset(profile.columns):
        profile["derived_wheelbase_length_ratio"] = profile["dim_wheelbase_in"] / profile["dim_length_in"].replace({0: np.nan})

    if {"dim_front_axle_track_in", "dim_rear_axle_track_in"}.issubset(profile.columns):
        profile["derived_avg_track_width_in"] = profile[["dim_front_axle_track_in", "dim_rear_axle_track_in"]].mean(axis=1)
        profile["derived_track_width_delta_in"] = profile["dim_rear_axle_track_in"] - profile["dim_front_axle_track_in"]

    return profile


def build_source_facts_long(inputs: dict[str, pd.DataFrame | None]) -> pd.DataFrame:
    """Clean source facts and preserve all source-level scraped facts."""
    facts = inputs.get("source_facts")
    if facts is None:
        return pd.DataFrame()

    facts = clean_frame_text(facts)

    required_cols = ["vehicle_id", "source_name", "source_url", "section_name", "section_slug", "fact_label", "fact_key", "fact_value"]
    for col in required_cols:
        if col not in facts.columns:
            facts[col] = np.nan

    facts["fact_id"] = facts.apply(
        lambda r: stable_id(r["vehicle_id"], r["source_name"], r["section_slug"], r["fact_key"], r["fact_value"], prefix="fact_"),
        axis=1,
    )

    facts["redlineiq_use_case"] = facts.apply(
        lambda r: classify_fact_use_case(r["fact_key"], r["fact_label"], r["section_name"]),
        axis=1,
    )

    facts["fact_statement"] = facts.apply(build_fact_statement, axis=1)

    ordered = [
        "fact_id",
        "vehicle_id",
        "redlineiq_use_case",
        "source_name",
        "source_url",
        "section_name",
        "section_slug",
        "fact_label",
        "fact_key",
        "fact_value",
        "fact_statement",
    ]
    return facts[ordered]


def build_context_facts(facts_long: pd.DataFrame) -> pd.DataFrame:
    """Filter long facts into facts likely useful for build/RAG logic."""
    if facts_long.empty:
        return facts_long

    useful_cases = {
        "suspension_setup_context",
        "wheel_tire_fitment_context",
        "brake_upgrade_context",
        "fueling_range_context",
        "maintenance_oil_context",
        "powertrain_context",
        "dimensions_aero_weight_context",
        "factory_performance_context",
    }

    context = facts_long[facts_long["redlineiq_use_case"].isin(useful_cases)].copy()

    # This is not a recommendation yet; it is structured context RedlineIQ can use to make suggestions later.
    context["recommendation_role"] = np.select(
        [
            context["redlineiq_use_case"].eq("suspension_setup_context"),
            context["redlineiq_use_case"].eq("wheel_tire_fitment_context"),
            context["redlineiq_use_case"].eq("brake_upgrade_context"),
            context["redlineiq_use_case"].eq("powertrain_context"),
            context["redlineiq_use_case"].eq("fueling_range_context"),
        ],
        [
            "baseline suspension architecture for upgrade compatibility",
            "baseline wheel/tire sizing for fitment checks",
            "baseline brake hardware for upgrade comparison",
            "baseline engine/powertrain for build path selection",
            "baseline fuel/range info for street-use context",
        ],
        default="supporting vehicle context",
    )

    return context.reset_index(drop=True)


def build_rag_context_chunks(profile: pd.DataFrame, facts_long: pd.DataFrame) -> pd.DataFrame:
    """Create simple RAG-ready context chunks from wide profile and source facts."""
    rows = []

    # One profile summary chunk per vehicle.
    for _, r in profile.iterrows():
        vehicle_id = r.get("vehicle_id", "")
        full_name = r.get("full_name", vehicle_id)
        make = r.get("make", "")
        model = r.get("model", "")
        trim = r.get("trim", "")

        summary_fields = []
        for col in [
            "generation", "platform_code", "body_style", "year_start", "year_end",
            "dim_wheelbase_in", "dim_length_in", "dim_width_in", "dim_height_in",
            "dim_curb_weight_lb", "chassis_front_tires", "chassis_rear_tires",
            "chassis_front_suspension", "chassis_rear_suspension",
            "powertrain_engine_code", "powertrain_aspiration", "powertrain_horsepower_hp",
            "powertrain_torque_lbft", "powertrain_drivetrain", "powertrain_transmission",
            "perf_top_speed_mph", "perf_acceleration_0_100_kmh_s",
            "derived_lb_per_hp", "derived_footprint_sq_ft",
        ]:
            if col in r.index and pd.notna(r[col]):
                summary_fields.append(f"{col}: {r[col]}")

        chunk_text = f"Vehicle profile for {full_name or vehicle_id} ({make} {model} {trim}). " + " | ".join(summary_fields)

        rows.append({
            "chunk_id": stable_id(vehicle_id, "vehicle_profile_summary", prefix="vehicle_chunk_"),
            "vehicle_id": vehicle_id,
            "source_type": "vehicle_profile_summary",
            "redlineiq_use_case": "structured_vehicle_profile",
            "title": f"Vehicle profile: {full_name or vehicle_id}",
            "url": r.get("source_url", np.nan),
            "chunk_text": chunk_text,
        })

    # One fact chunk per source fact.
    if not facts_long.empty:
        for _, r in facts_long.iterrows():
            rows.append({
                "chunk_id": stable_id(r["fact_id"], "source_fact", prefix="vehicle_fact_chunk_"),
                "vehicle_id": r["vehicle_id"],
                "source_type": "vehicle_source_fact",
                "redlineiq_use_case": r["redlineiq_use_case"],
                "title": f"{r['section_name']} | {r['fact_label']}",
                "url": r["source_url"],
                "chunk_text": r["fact_statement"],
            })

    return pd.DataFrame(rows)


def build_etl_summary(inputs: dict[str, pd.DataFrame | None], profile: pd.DataFrame, facts: pd.DataFrame, context: pd.DataFrame, chunks: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, df in inputs.items():
        rows.append({
            "artifact": f"input_{name}",
            "rows": 0 if df is None else len(df),
            "columns": 0 if df is None else len(df.columns),
            "notes": "missing" if df is None else "loaded",
        })

    rows.extend([
        {"artifact": "output_vehicle_profile_wide_cleaned", "rows": len(profile), "columns": len(profile.columns), "notes": "one row per vehicle; all structured tables outer-joined"},
        {"artifact": "output_vehicle_source_facts_cleaned_long", "rows": len(facts), "columns": len(facts.columns), "notes": "all scraped source facts preserved in long format"},
        {"artifact": "output_vehicle_build_context_facts", "rows": len(context), "columns": len(context.columns), "notes": "subset of facts useful for build recommendations/RAG"},
        {"artifact": "output_vehicle_rag_context_chunks", "rows": len(chunks), "columns": len(chunks.columns), "notes": "chunks ready for RedlineIQ retrieval/embedding"},
    ])

    return pd.DataFrame(rows)


def run_etl(input_dir: Path, output_dir: Path) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    inputs = load_inputs(input_dir)

    profile = build_wide_vehicle_profile(inputs)
    facts_long = build_source_facts_long(inputs)
    context_facts = build_context_facts(facts_long)
    rag_chunks = build_rag_context_chunks(profile, facts_long)
    summary = build_etl_summary(inputs, profile, facts_long, context_facts, rag_chunks)

    output_paths = {
        "vehicle_profile_wide_cleaned": output_dir / "vehicle_profile_wide_cleaned.csv",
        "vehicle_source_facts_cleaned_long": output_dir / "vehicle_source_facts_cleaned_long.csv",
        "vehicle_build_context_facts": output_dir / "vehicle_build_context_facts.csv",
        "vehicle_rag_context_chunks": output_dir / "vehicle_rag_context_chunks.csv",
        "vehicle_profile_etl_summary": output_dir / "vehicle_profile_etl_summary.csv",
    }

    profile.to_csv(output_paths["vehicle_profile_wide_cleaned"], index=False)
    facts_long.to_csv(output_paths["vehicle_source_facts_cleaned_long"], index=False)
    context_facts.to_csv(output_paths["vehicle_build_context_facts"], index=False)
    rag_chunks.to_csv(output_paths["vehicle_rag_context_chunks"], index=False)
    summary.to_csv(output_paths["vehicle_profile_etl_summary"], index=False)

    print(json.dumps({k: str(v) for k, v in output_paths.items()}, indent=2))
    print("\nProfile shape:", profile.shape)
    print("Facts preserved:", facts_long.shape)
    print("Build/RAG context facts:", context_facts.shape)
    print("RAG chunks:", rag_chunks.shape)

    if not context_facts.empty:
        print("\nContext fact use cases:")
        print(context_facts["redlineiq_use_case"].value_counts().to_string())

    return output_paths

In [10]:
def read_optional_csv(path: Path):
    if path.exists():
        return pd.read_csv(path)
    return None


def read_csv_if_exists(path):
    """Read a CSV if it exists, otherwise return None."""
    if path.exists():
        return pd.read_csv(path)
    return None


def first_existing(input_dir: Path, candidates: Iterable[str]) -> Path | None:
    """Return the first existing file path from a list of candidate filenames."""
    for name in candidates:
        path = input_dir / name
        if path.exists():
            return path
    return None


def clean_pipe_text(value):
    """Normalize scraped text like '| Coupé' or '| RWD' into 'Coupé' / 'RWD'."""
    if pd.isna(value):
        return np.nan
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip("|").strip()
    text = re.sub(r"\s*\|\s*", " | ", text)
    return text if text else np.nan


def numeric_clean(series: pd.Series) -> pd.Series:
    """Convert numeric-like text to numeric values."""
    return pd.to_numeric(series, errors="coerce")


def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace({0: np.nan})
    return numerator / denominator


def add_unit_delta(df: pd.DataFrame, inch_col: str, metric_col: str, factor: float = 2.54) -> pd.Series:
    """Return percent delta between metric column and inch_col * factor."""
    expected_metric = df[inch_col] * factor
    return ((df[metric_col] - expected_metric).abs() / expected_metric.replace({0: np.nan}))


def build_quality_notes(row: pd.Series) -> str:
    notes: list[str] = []

    required = [
        "vehicle_id",
        "wheelbase_in",
        "length_in",
        "width_in",
        "height_in",
        "curb_weight_lb",
    ]

    missing_required = [c for c in required if c in row.index and pd.isna(row[c])]
    if missing_required:
        notes.append("missing_required=" + ",".join(missing_required))

    if "wheelbase_length_ratio" in row.index and pd.notna(row["wheelbase_length_ratio"]):
        if not 0.45 <= row["wheelbase_length_ratio"] <= 0.75:
            notes.append("wheelbase_length_ratio_outside_expected_range")

    if "curb_weight_lb" in row.index and pd.notna(row["curb_weight_lb"]):
        if row["curb_weight_lb"] < 1000 or row["curb_weight_lb"] > 8000:
            notes.append("curb_weight_outside_expected_range")

    if "length_in" in row.index and pd.notna(row["length_in"]):
        if row["length_in"] < 100 or row["length_in"] > 250:
            notes.append("length_outside_expected_range")

    # Unit consistency flags are created dynamically by clean_dimensions().
    delta_cols = [c for c in row.index if c.endswith("_unit_delta_pct")]
    bad_deltas = [c for c in delta_cols if pd.notna(row[c]) and row[c] > 0.03]
    if bad_deltas:
        notes.append("unit_delta_gt_3pct=" + ",".join(bad_deltas))

    return "; ".join(notes) if notes else "ok"

In [13]:
def load_inputs(input_dir: Path) -> dict[str, pd.DataFrame | None]:
    chassis_path = first_existing(
        input_dir,
        ["vehicle_chassis_specs.csv", "vehicle_chassis_specs(1).csv"],
    )

    return {
        "dimensions": read_csv_if_exists(input_dir / "vehicle_dimensions.csv"),
        "vehicles": read_csv_if_exists(input_dir / "vehicles.csv"),
        "chassis": read_csv_if_exists(chassis_path) if chassis_path else None,
        "powertrain": read_csv_if_exists(input_dir / "vehicle_powertrain_specs.csv"),
        "performance": read_csv_if_exists(input_dir / "vehicle_factory_performance.csv"),
    }


def clean_dimensions(dimensions: pd.DataFrame) -> pd.DataFrame:
    df = dimensions.copy()

    # Normalize column names in case source files change slightly.
    df.columns = [c.strip().lower() for c in df.columns]

    # Trim ID/text fields.
    text_cols = df.select_dtypes(include=["object"]).columns.tolist()
    for col in text_cols:
        df[col] = df[col].apply(clean_pipe_text)

    # Convert all non-ID columns that look numeric.
    for col in df.columns:
        if col != "vehicle_id":
            df[col] = numeric_clean(df[col])

    # Deduplicate by vehicle_id, preferring the row with the most populated values.
    if "vehicle_id" in df.columns:
        df["_non_null_count"] = df.notna().sum(axis=1)
        df = (
            df.sort_values(["vehicle_id", "_non_null_count"], ascending=[True, False])
            .drop_duplicates(subset=["vehicle_id"], keep="first")
            .drop(columns=["_non_null_count"])
            .reset_index(drop=True)
        )

    # Unit consistency checks.
    unit_pairs = [
        ("wheelbase_in", "wheelbase_cm", 2.54),
        ("length_in", "length_cm", 2.54),
        ("width_in", "width_cm", 2.54),
        ("width_with_mirrors_in", "width_with_mirrors_cm", 2.54),
        ("height_in", "height_cm", 2.54),
        ("front_axle_track_in", "front_axle_track_cm", 2.54),
        ("rear_axle_track_in", "rear_axle_track_cm", 2.54),
        ("ground_clearance_in", "ground_clearance_cm", 2.54),
        ("curb_weight_lb", "curb_weight_kg", 0.45359237),
        ("trunk_capacity_cuft", "trunk_capacity_l", 28.3168466),
    ]

    for imperial_col, metric_col, factor in unit_pairs:
        if imperial_col in df.columns and metric_col in df.columns:
            delta_col = f"{imperial_col}_to_{metric_col}_unit_delta_pct"
            df[delta_col] = add_unit_delta(df, imperial_col, metric_col, factor)

    # RedlineIQ useful derived features.
    if {"length_in", "width_in"}.issubset(df.columns):
        df["footprint_sq_ft"] = (df["length_in"] * df["width_in"]) / 144

    if {"width_in", "height_in"}.issubset(df.columns):
        df["frontal_area_proxy_sq_ft"] = (df["width_in"] * df["height_in"]) / 144

    if {"wheelbase_in", "length_in"}.issubset(df.columns):
        df["wheelbase_length_ratio"] = safe_divide(df["wheelbase_in"], df["length_in"])

    if {"front_axle_track_in", "rear_axle_track_in"}.issubset(df.columns):
        df["avg_track_width_in"] = df[["front_axle_track_in", "rear_axle_track_in"]].mean(axis=1)
        df["track_width_delta_in"] = df["rear_axle_track_in"] - df["front_axle_track_in"]

    if {"curb_weight_lb", "footprint_sq_ft"}.issubset(df.columns):
        df["curb_weight_per_footprint_lb_sqft"] = safe_divide(df["curb_weight_lb"], df["footprint_sq_ft"])

    df["dimension_quality_notes"] = df.apply(build_quality_notes, axis=1)
    df["dimension_quality_status"] = np.where(df["dimension_quality_notes"].eq("ok"), "pass", "review")

    return df


def enrich_dimensions(cleaned: pd.DataFrame, inputs: dict[str, pd.DataFrame | None]) -> pd.DataFrame:
    df = cleaned.copy()

    vehicles = inputs.get("vehicles")
    if vehicles is not None and "vehicle_id" in vehicles.columns:
        vehicle_cols = [
            c for c in [
                "vehicle_id",
                "make",
                "model",
                "trim",
                "full_name",
                "generation",
                "platform_code",
                "body_style",
                "doors",
                "seats",
                "year_start",
                "year_end",
                "source_name",
                "source_url",
            ]
            if c in vehicles.columns
        ]
        vehicles = vehicles[vehicle_cols].copy()
        for col in vehicles.select_dtypes(include=["object"]).columns:
            vehicles[col] = vehicles[col].apply(clean_pipe_text)
        df = df.merge(vehicles, on="vehicle_id", how="left", suffixes=("", "_vehicle"))

    chassis = inputs.get("chassis")
    if chassis is not None and "vehicle_id" in chassis.columns:
        chassis_cols = [
            c for c in [
                "vehicle_id",
                "front_tires",
                "rear_tires",
                "turning_circle_ft",
                "turning_circle_m",
                "front_suspension",
                "rear_suspension",
            ]
            if c in chassis.columns
        ]
        chassis = chassis[chassis_cols].copy()
        for col in chassis.select_dtypes(include=["object"]).columns:
            chassis[col] = chassis[col].apply(clean_pipe_text)
        df = df.merge(chassis, on="vehicle_id", how="left")

    powertrain = inputs.get("powertrain")
    if powertrain is not None and "vehicle_id" in powertrain.columns:
        powertrain_cols = [
            c for c in [
                "vehicle_id",
                "engine_code",
                "aspiration",
                "horsepower_hp",
                "torque_lbft",
                "drivetrain",
                "transmission",
            ]
            if c in powertrain.columns
        ]
        powertrain = powertrain[powertrain_cols].copy()
        for col in powertrain.select_dtypes(include=["object"]).columns:
            powertrain[col] = powertrain[col].apply(clean_pipe_text)
        df = df.merge(powertrain, on="vehicle_id", how="left")

    performance = inputs.get("performance")
    if performance is not None and "vehicle_id" in performance.columns:
        performance_cols = [
            c for c in [
                "vehicle_id",
                "top_speed_mph",
                "top_speed_kmh",
                "acceleration_0_100_kmh_s",
            ]
            if c in performance.columns
        ]
        df = df.merge(performance[performance_cols], on="vehicle_id", how="left")

    # Derived cross-table features.
    if {"curb_weight_lb", "horsepower_hp"}.issubset(df.columns):
        df["lb_per_hp"] = safe_divide(df["curb_weight_lb"], df["horsepower_hp"])
        df["hp_per_1000_lb"] = safe_divide(df["horsepower_hp"] * 1000, df["curb_weight_lb"])

    # Reorder important columns first.
    preferred_first = [
        "vehicle_id",
        "make",
        "model",
        "trim",
        "full_name",
        "generation",
        "platform_code",
        "body_style",
        "year_start",
        "year_end",
        "wheelbase_in",
        "length_in",
        "width_in",
        "height_in",
        "curb_weight_lb",
        "footprint_sq_ft",
        "frontal_area_proxy_sq_ft",
        "wheelbase_length_ratio",
        "avg_track_width_in",
        "track_width_delta_in",
        "curb_weight_per_footprint_lb_sqft",
        "front_tires",
        "rear_tires",
        "turning_circle_ft",
        "engine_code",
        "horsepower_hp",
        "torque_lbft",
        "lb_per_hp",
        "hp_per_1000_lb",
        "top_speed_mph",
        "acceleration_0_100_kmh_s",
        "dimension_quality_status",
        "dimension_quality_notes",
        "source_name",
        "source_url",
    ]

    ordered = [c for c in preferred_first if c in df.columns]
    remaining = [c for c in df.columns if c not in ordered]
    df = df[ordered + remaining]

    return df


def build_eda_summary(raw_dimensions: pd.DataFrame, cleaned: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []

    for name, df in [("raw_dimensions", raw_dimensions), ("cleaned_dimensions", cleaned)]:
        summary_rows.append({
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "duplicate_vehicle_ids": int(df.duplicated(subset=["vehicle_id"]).sum()) if "vehicle_id" in df.columns else np.nan,
            "missing_vehicle_id": int(df["vehicle_id"].isna().sum()) if "vehicle_id" in df.columns else np.nan,
            "numeric_columns": int(len(df.select_dtypes(include=["number"]).columns)),
            "object_columns": int(len(df.select_dtypes(include=["object"]).columns)),
        })

    return pd.DataFrame(summary_rows)


def build_quality_report(cleaned: pd.DataFrame) -> pd.DataFrame:
    quality_cols = [
        c for c in cleaned.columns
        if c in ["vehicle_id", "full_name", "dimension_quality_status", "dimension_quality_notes"]
        or c.endswith("_unit_delta_pct")
    ]
    return cleaned[quality_cols].copy()



def run_etl(input_dir: Path, output_dir: Path) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    inputs = load_inputs(input_dir)
    dimensions = inputs.get("dimensions")

    if dimensions is None:
        raise FileNotFoundError(f"vehicle_dimensions.csv not found in {input_dir}")

    cleaned_base = clean_dimensions(dimensions)
    cleaned = enrich_dimensions(cleaned_base, inputs)

    cleaned_path = output_dir / "vehicle_dimensions_cleaned.csv"
    example_path = output_dir / "vehicle_dimensions_cleaned_example.csv"
    eda_summary_path = output_dir / "vehicle_dimensions_eda_summary.csv"
    quality_report_path = output_dir / "vehicle_dimensions_quality_report.csv"

    cleaned.to_csv(cleaned_path, index=False)
    cleaned.head(10).to_csv(example_path, index=False)
    build_eda_summary(dimensions, cleaned).to_csv(eda_summary_path, index=False)
    build_quality_report(cleaned).to_csv(quality_report_path, index=False)

    paths = {
        "cleaned": cleaned_path,
        "example": example_path,
        "eda_summary": eda_summary_path,
        "quality_report": quality_report_path,
    }

    print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))
    print("\nCleaned shape:", cleaned.shape)
    print("Quality status counts:")
    print(cleaned["dimension_quality_status"].value_counts(dropna=False).to_string())

    return paths

In [ ]:
INPUT_DIR = Path('../backend/app/data/ingestion/raw/vehicle_specs_raw/gr_supra_spec')
OUTPUT_DIR = Path('../backend/app/data/processed/vehicle_specs_cleaned')


PATHS = {
    'dimensions': INPUT_DIR / 'vehicle_dimensions.csv',
    'vehicles': INPUT_DIR / 'vehicles.csv',
    'chassis_primary': INPUT_DIR / 'vehicle_chassis_specs.csv'
    'powertrain': INPUT_DIR / 'vehicle_powertrain_specs.csv',
    'performance': INPUT_DIR / 'vehicle_factory_performance.csv',
}


In [25]:
dimensions = pd.read_csv(PATHS['dimensions'])
vehicles = read_optional_csv(PATHS['vehicles'])
chassis = read_optional_csv(PATHS['chassis_primary'])
powertrain = read_optional_csv(PATHS['powertrain'])
performance = read_optional_csv(PATHS['performance'])

datasets = {
    'dimensions': dimensions,
    'vehicles': vehicles,
    'chassis': chassis,
    'powertrain': powertrain,
    'performance': performance,
}

for name, df in datasets.items():
    if df is None:
        print(f'{name}: missing')
    else:
        print(f'{name}: {df.shape[0]} rows x {df.shape[1]} columns')
        display(df.head(2))


dimensions: 1 rows x 23 columns


,vehicle_id,wheelbase_in,wheelbase_cm,length_in,length_cm,width_in,width_cm,width_with_mirrors_in,width_with_mirrors_cm,height_in,...,rear_axle_track_in,rear_axle_track_cm,ground_clearance_in,ground_clearance_cm,drag_coefficient_cx,curb_weight_lb,curb_weight_kg,weight_power_ratio_kg_per_hp,trunk_capacity_cuft,trunk_capacity_l
0,toyota_gr_supra_3_0_auto_2024_2025,97.24,247.0,172.4,437.9,72.99,185.4,80.12,203.5,50.87,...,62.56,158.9,4.53,11.5,NaN,3461.0,1570.0,4.6,10.2,290.0


vehicles: 1 rows x 14 columns


,vehicle_id,make,model,trim,full_name,generation,platform_code,body_style,doors,seats,year_start,year_end,source_name,source_url
0,toyota_gr_supra_3_0_auto_2024_2025,Toyota,GR Supra,3.0 Auto,Toyota GR Supra 3.0 Auto,| GR Supra Mk5 (J29),J29,| Coupé,3,2,2024,2025,Ultimate Specs,https://www.ultimatespecs.com/car-specs/Toyota...


chassis: 1 rows x 13 columns


,vehicle_id,front_brake_type,front_brake_disc_in,front_brake_disc_mm,rear_brake_type,rear_brake_disc_in,rear_brake_disc_mm,front_tires,rear_tires,turning_circle_ft,turning_circle_m,front_suspension,rear_suspension
0,toyota_gr_supra_3_0_auto_2024_2025,Vented Discs,NaN,NaN,Vented Discs,NaN,NaN,| 255/35 R19,| 275/35 R19,36.1,11.0,| Independent. MacPherson. Coil Springs. Anti-...,| Multi-link. Coil Springs. Anti-roll bar.


powertrain: 1 rows x 30 columns


,vehicle_id,engine_layout,engine_code,fuel_type,fuel_system,lubrication_raw,engine_oil_viscosity,engine_oem_oil_specs,engine_oil_api_acea,engine_oil_capacity_l,...,compression_ratio,horsepower_hp,horsepower_ps,horsepower_kw,horsepower_rpm_band,torque_lbft,torque_nm,torque_rpm_band,drivetrain,transmission
0,toyota_gr_supra_3_0_auto_2024_2025,| Inline 6,| BMW B58B30C,| Petrol,| Direct Injection,| See oil information | Toyota GR Supra 3.0 Au...,NaN,NaN,NaN,NaN,...,11.0,335.0,340.0,250.0,5000-6500 rpm,368.0,500.0,1600-4500 rpm,| RWD,| 8 speed Automatic


performance: 1 rows x 4 columns


,vehicle_id,top_speed_mph,top_speed_kmh,acceleration_0_100_kmh_s
0,toyota_gr_supra_3_0_auto_2024_2025,155.0,250.0,4.3


## EDA: Missing, duplicates, and sumamry

In [8]:
missing = (
    dimensions.isna().sum()
    .rename('missing_count')
    .to_frame()
)
missing['missing_pct'] = missing['missing_count'] / len(dimensions)
missing = missing.sort_values('missing_count', ascending=False)
display(missing)

if 'vehicle_id' in dimensions.columns:
    print('Duplicate vehicle_id rows:', dimensions.duplicated(subset=['vehicle_id']).sum())

display(dimensions.describe(include='all').T)

,missing_count,missing_pct
drag_coefficient_cx,1,1.0
vehicle_id,0,0.0
front_axle_track_cm,0,0.0
trunk_capacity_cuft,0,0.0
weight_power_ratio_kg_per_hp,0,0.0
curb_weight_kg,0,0.0
curb_weight_lb,0,0.0
ground_clearance_cm,0,0.0
ground_clearance_in,0,0.0
rear_axle_track_cm,0,0.0


Duplicate vehicle_id rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
vehicle_id,1,1,toyota_gr_supra_3_0_auto_2024_2025,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wheelbase_in,1.0,NaN,NaN,NaN,97.24,NaN,97.24,97.24,97.24,97.24,97.24
wheelbase_cm,1.0,NaN,NaN,NaN,247.0,NaN,247.0,247.0,247.0,247.0,247.0
length_in,1.0,NaN,NaN,NaN,172.4,NaN,172.4,172.4,172.4,172.4,172.4
length_cm,1.0,NaN,NaN,NaN,437.9,NaN,437.9,437.9,437.9,437.9,437.9
width_in,1.0,NaN,NaN,NaN,72.99,NaN,72.99,72.99,72.99,72.99,72.99
width_cm,1.0,NaN,NaN,NaN,185.4,NaN,185.4,185.4,185.4,185.4,185.4
width_with_mirrors_in,1.0,NaN,NaN,NaN,80.12,NaN,80.12,80.12,80.12,80.12,80.12
width_with_mirrors_cm,1.0,NaN,NaN,NaN,203.5,NaN,203.5,203.5,203.5,203.5,203.5
height_in,1.0,NaN,NaN,NaN,50.87,NaN,50.87,50.87,50.87,50.87,50.87


## Unit consistency check
checks whether inch/cm, lb/kg, and cuft/liter fields agree within a small tolerance
- checks catch scraper errors, source inconsistencies, and bad unit conversions

In [ ]:
unit_pairs = [
    #values in inches should match with cm values when cm is converted to inches (2.54 cm per inch)
    ('wheelbase_in', 'wheelbase_cm', 2.54),
    ('length_in', 'length_cm', 2.54),
    ('width_in', 'width_cm', 2.54),
    ('width_with_mirrors_in', 'width_with_mirrors_cm', 2.54),
    ('height_in', 'height_cm', 2.54),
    ('front_axle_track_in', 'front_axle_track_cm', 2.54),
    ('rear_axle_track_in', 'rear_axle_track_cm', 2.54),
    ('ground_clearance_in', 'ground_clearance_cm', 2.54),
    #check weights in pounds vs kg (1 lb = 0.45359237 kg)
    ('curb_weight_lb', 'curb_weight_kg', 0.45359237),
    #checks trunk cubic feet vs liters (1 cu ft = 28.317 liters)
    ('trunk_capacity_cuft', 'trunk_capacity_l', 28.317),
]

unit_check_rows = []
for imperial_col, metric_col, factor in unit_pairs:
    if imperial_col in dimensions.columns and metric_col in dimensions.columns:
        expected = dimensions[imperial_col] * factor
        delta_pct = ((dimensions[metric_col] - expected).abs() / expected.replace({0: np.nan}))
        unit_check_rows.append({
            'source_col': imperial_col,
            'target_col': metric_col,
            'max_delta_pct': float(delta_pct.max(skipna=True)) if delta_pct.notna().any() else np.nan,
            'mean_delta_pct': float(delta_pct.mean(skipna=True)) if delta_pct.notna().any() else np.nan,
        })

unit_checks = pd.DataFrame(unit_check_rows)
display(unit_checks)

,source_col,target_col,max_delta_pct,mean_delta_pct
0,wheelbase_in,wheelbase_cm,0.000042,0.000042
1,length_in,length_cm,0.000009,0.000009
2,width_in,width_cm,0.000029,0.000029
3,width_with_mirrors_in,width_with_mirrors_cm,0.000024,0.000024
4,height_in,height_cm,0.000076,0.000076
5,front_axle_track_in,front_axle_track_cm,0.000065,0.000065
6,rear_axle_track_in,rear_axle_track_cm,0.000015,0.000015
7,ground_clearance_in,ground_clearance_cm,0.000539,0.000539
8,curb_weight_lb,curb_weight_kg,0.000074,0.000074
9,trunk_capacity_cuft,trunk_capacity_l,0.004044,0.004044


In [18]:
outputs = run_etl(INPUT_DIR, OUTPUT_DIR)


{
  "cleaned": "../backend/app/data/processed/vehicle_specs_cleaned/vehicle_dimensions_cleaned.csv",
  "example": "../backend/app/data/processed/vehicle_specs_cleaned/vehicle_dimensions_cleaned_example.csv",
  "eda_summary": "../backend/app/data/processed/vehicle_specs_cleaned/vehicle_dimensions_eda_summary.csv",
  "quality_report": "../backend/app/data/processed/vehicle_specs_cleaned/vehicle_dimensions_quality_report.csv"
}

Cleaned shape: (1, 71)
Quality status counts:
dimension_quality_status
pass    1


/var/folders/8n/k4ymc8nx1q5420j8bz49s4440000gn/T/ipykernel_5675/2350165649.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=["object"]).columns.tolist()
/var/folders/8n/k4ymc8nx1q5420j8bz49s4440000gn/T/ipykernel_5675/2350165649.py:109: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See 

In [21]:
outputs["cleaned"]

PosixPath('../backend/app/data/processed/vehicle_specs_cleaned/vehicle_dimensions_cleaned.csv')